<a href="https://colab.research.google.com/github/Manmanfr/UIS-Render/blob/main/Segmentacion%20con%20LANGSAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Primero segmentamos el mapa con SAM normal

In [ ]:
import os
import torch
from samgeo import SamGeo
from samgeo.common import split_raster, merge_rasters, raster_to_vector

# 1. Rutas principales
imagen_uis = "/content/drive/MyDrive/Procesamiento_SAM/UIS_Central.tif"
carpeta_base = "/content/drive/MyDrive/Procesamiento_SAM"

# Carpetas temporales para guardar los recortes y las máscaras
tile_dir = os.path.join(carpeta_base, "tiles_entrada")
mask_dir = os.path.join(carpeta_base, "tiles_salida")
salida_tif_final = os.path.join(carpeta_base, "segmentacion_final.tif")
salida_shp = os.path.join(carpeta_base, "poligonos_uis.shp")

os.makedirs(tile_dir, exist_ok=True)
os.makedirs(mask_dir, exist_ok=True)

# Dividir en mosaicos
split_raster(imagen_uis, out_dir=tile_dir, tile_size=(1500, 1500), overlap=50)

sam = SamGeo(model_type="vit_b", device="cuda")
tiles = [f for f in os.listdir(tile_dir) if f.endswith('.tif')]
print(f"Total de recuadros a procesar: {len(tiles)}")

for i, tile in enumerate(tiles):
    print(f"Segmentando fragmento {i+1}/{len(tiles)}: {tile}")
    in_path = os.path.join(tile_dir, tile)
    out_path = os.path.join(mask_dir, tile)

    sam.generate(in_path, out_path)
    torch.cuda.empty_cache()

print("3. Uniendo los pedazos segmentados...")
merge_rasters(mask_dir, salida_tif_final)

print("4. Vectorizando el mapa completo a Shapefile...")
raster_to_vector(salida_tif_final, salida_shp)


#Ya con el mapa segmentado, usamos langSAM para segmentar cosas de nuestro interes


In [ ]:
import os
import torch
from samgeo.text_sam import LangSAM
from samgeo.common import split_raster, merge_rasters, raster_to_vector

# Rutas
imagen_uis = "/content/drive/MyDrive/Procesamiento_SAM/UIS_Central.tif"
carpeta_base = "/content/drive/MyDrive/Procesamiento_SAM"

# Carpeta con la imagen ya segmentada o si esta carpeta no esta procedemos a segmentar la imagen
tile_dir = os.path.join(carpeta_base, "tiles_entrada")
os.makedirs(tile_dir, exist_ok=True)

if not os.listdir(tile_dir):
    split_raster(imagen_uis, out_dir=tile_dir, tile_size=(1500, 1500), overlap=50)

# Categorias que vamos a buscar
categorias = {
    "edificios": ("building roof", 0.45, 0.45),
    "arboles": ("tree", 0.30, 0.30),
    "caminos": ("asphalt road or paved path", 0.25, 0.25)
}

sam = LangSAM()

tiles = [f for f in os.listdir(tile_dir) if f.endswith('.tif')]
print(f"Total de fragmentos a procesar por categoría: {len(tiles)}")

# Bucle principal
for nombre_capa, (prompt, box_thresh, text_thresh) in categorias.items():
    print(f"PROCESANDO CATEGORÍA: {nombre_capa.upper()}")
    print(f"Prompt: '{prompt}' | Umbrales: Box={box_thresh}, Text={text_thresh}")
    mask_dir_clase = os.path.join(carpeta_base, f"tiles_salida_{nombre_capa}")
    os.makedirs(mask_dir_clase, exist_ok=True)

    salida_tif_final = os.path.join(carpeta_base, f"segmentacion_{nombre_capa}.tif")
    salida_shp = os.path.join(carpeta_base, f"poligonos_{nombre_capa}.shp")

    # Segmentar cada fragmento usando el prompt de la categoría actual
    for i, tile in enumerate(tiles):
        print(f"[{nombre_capa.upper()}] Analizando fragmento {i+1}/{len(tiles)}: {tile}")
        in_path = os.path.join(tile_dir, tile)
        out_path = os.path.join(mask_dir_clase, tile)

        sam.predict(
            image=in_path,
            text_prompt=prompt,
            box_threshold=box_thresh,
            text_threshold=text_thresh,
            output=out_path
        )
        torch.cuda.empty_cache()

    print(f"Uniendo fragmentos de {nombre_capa}...")
    merge_rasters(mask_dir_clase, salida_tif_final)

    print(f"Vectorizando {nombre_capa} a Shapefile...")
    raster_to_vector(salida_tif_final, salida_shp)

